# Four-stage LIMO rosbag plots

This notebook reads the `rosbag2_2026_08_21` experiment and plots ground-truth velocity, learned/model Hamiltonian gradients, and both affine-policy coefficients. Times are measured from the first learning sample.

In [ ]:
from collections import defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import rosbag2_py
from rclpy.serialization import deserialize_message
from rosidl_runtime_py.utilities import get_message
from scipy.io import savemat

plt.style.use('ggplot')
plt.rcParams.update({'figure.figsize': (11, 4.5), 'figure.dpi': 120})

In [ ]:
# Locate the package whether the notebook is launched from the scripts folder
# or from the workspace root.
candidates = [
    Path.cwd().parent / 'bags',
    Path.cwd() / 'limo_four_stage_learning' / 'bags',
    Path('/home/panos/code/limo_public_ws/src/limo_four_stage_learning/bags'),
]
bags_root = next(path for path in candidates if path.is_dir())
exact_path = bags_root / 'rosbag2_2026_08_21'
matches = [exact_path] if exact_path.is_dir() else sorted(
    bags_root.glob('rosbag2_2026_08_21*')
)
if len(matches) != 1:
    raise RuntimeError(f'Expected one 2026-08-21 bag, found: {matches}')
bag_path = matches[0]
print('Reading:', bag_path)

topics = {
    '/limo_1/ground_truth',
    '/four_stage_learning/learned_gradient',
    '/four_stage_learning/model_gradient',
    '/four_stage_learning/policy',
}

reader = rosbag2_py.SequentialReader()
reader.open(
    rosbag2_py.StorageOptions(uri=str(bag_path), storage_id='sqlite3'),
    rosbag2_py.ConverterOptions('', ''),
)
topic_types = {item.name: item.type for item in reader.get_all_topics_and_types()}
missing = topics - topic_types.keys()
if missing:
    raise KeyError(f'Missing required topics: {sorted(missing)}')

records = defaultdict(list)
while reader.has_next():
    topic, serialized, timestamp_ns = reader.read_next()
    if topic not in topics:
        continue
    message_type = get_message(topic_types[topic])
    message = deserialize_message(serialized, message_type)
    if topic == '/limo_1/ground_truth':
        value = message.twist.twist.linear.x
    elif topic == '/four_stage_learning/policy':
        if len(message.data) < 2:
            raise ValueError('Policy message does not contain [k_e, k_0]')
        value = (message.data[0], message.data[1])
    else:
        value = message.data
    records[topic].append((timestamp_ns, value))

experiment_start_ns = records['/four_stage_learning/learned_gradient'][0][0]
def unpack(topic):
    entries = records[topic]
    time = np.array([(stamp - experiment_start_ns) * 1e-9 for stamp, _ in entries])
    values = np.asarray([value for _, value in entries])
    return time, values

velocity_t, velocity = unpack('/limo_1/ground_truth')
learned_t, learned_gradient = unpack('/four_stage_learning/learned_gradient')
model_t, model_gradient = unpack('/four_stage_learning/model_gradient')
policy_t, policy = unpack('/four_stage_learning/policy')

# Discard ground-truth samples recorded before the controller began.
experiment_end = max(learned_t[-1], model_t[-1], policy_t[-1])
mask = (velocity_t >= 0.0) & (velocity_t <= experiment_end)
velocity_t, velocity = velocity_t[mask], velocity[mask]
print({topic: len(values) for topic, values in records.items()})

In [ ]:
fig, ax = plt.subplots()
ax.plot(velocity_t, velocity, color='tab:blue', linewidth=1.5)
ax.set(title='LIMO ground-truth velocity', xlabel='Experiment time [s]', ylabel='Velocity [m/s]')
fig.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots()
ax.plot(learned_t, learned_gradient, label='Learned gradient', linewidth=1.2)
ax.plot(model_t, model_gradient, label='Model gradient', linewidth=1.5, alpha=0.85)
ax.set(title='Hamiltonian gradients', xlabel='Experiment time [s]', ylabel=r'$\nabla_u H$')
ax.legend()
fig.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots()
ax.plot(policy_t, policy[:, 0], color='tab:green', linewidth=1.5)
ax.set(title='Policy slope', xlabel='Experiment time [s]', ylabel=r'$k_e$ (policy.data[0])')
fig.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots()
ax.plot(policy_t, policy[:, 1], color='tab:orange', linewidth=1.5)
ax.set(title='Policy intercept', xlabel='Experiment time [s]', ylabel=r'$k_0$ (policy.data[1])')
fig.tight_layout()
plt.show()

In [ ]:
# Export one MATLAB data file per figure.
scripts_candidates = [
    Path.cwd() if Path.cwd().name == 'scripts' else None,
    Path.cwd() / 'limo_four_stage_learning' / 'scripts',
    Path('/home/panos/code/limo_public_ws/src/limo_four_stage_learning/scripts'),
]
scripts_dir = next(path for path in scripts_candidates if path and path.is_dir())
mat_dir = scripts_dir / 'mat_data'
mat_dir.mkdir(parents=True, exist_ok=True)
savemat(mat_dir / 'ground_truth_velocity.mat', {
    'time': velocity_t, 'velocity': velocity,
})
savemat(mat_dir / 'hamiltonian_gradients.mat', {
    'learned_time': learned_t, 'learned_gradient': learned_gradient,
    'model_time': model_t, 'model_gradient': model_gradient,
})
savemat(mat_dir / 'policy_slope.mat', {
    'time': policy_t, 'policy_slope': policy[:, 0],
})
savemat(mat_dir / 'policy_intercept.mat', {
    'time': policy_t, 'policy_intercept': policy[:, 1],
})
print('Saved MATLAB files to:', mat_dir)